# 2. Reactant descriptors and SVO areas

Input: `all_smiles.csv` and the optimized reactant MOL files from part 1. Output: `Descriptors.csv`, `Descriptors_.csv`, and `Descriptors_.pkl` in `SMILES_DIR`. The later pair-generation stage currently uses a separately named descriptor map; verify its provenance before substituting this pickle.

Saved outputs were cleared in this split. The archived source notebook retains the historical outputs and execution counts. Set the job paths before running any calculation.


In [ ]:
from pathlib import Path
import os
import sys

# Resolve the repository root from the kernel working directory, or set CYCLOADD_REPO_ROOT.
repo_root_env = os.environ.get("CYCLOADD_REPO_ROOT")
if repo_root_env:
    REPO_ROOT = Path(repo_root_env).expanduser().resolve()
    if not (REPO_ROOT / "Code").is_dir() or not (REPO_ROOT / "Data").is_dir():
        raise RuntimeError(f"CYCLOADD_REPO_ROOT is not a repository root: {REPO_ROOT}")
else:
    REPO_ROOT = next(
        (parent for parent in (Path.cwd(), *Path.cwd().parents)
         if (parent / "Code").is_dir() and (parent / "Data").is_dir()),
        None,
    )
if REPO_ROOT is None:
    raise RuntimeError("Start Jupyter in this repository or set CYCLOADD_REPO_ROOT")
if str(REPO_ROOT) not in sys.path:
    sys.path.insert(0, str(REPO_ROOT))
os.chdir(REPO_ROOT)

import pickle
import numpy as np
import pandas as pd
from rdkit import Chem
from tqdm import tqdm
from Code import cycle_process, main, model_train

SMILES_DIR = os.environ.get("CYCLOADD_SMILES_DIR", r"E:\work\Secondary_Selection\Diene_Ene_Smiles")


# 2. Get Descriptors for Reactants

In [ ]:
ALL_PROPERTIES_diene = ["diene_charge_a", "diene_charge_b", "diene_charge_c", "diene_charge_d",
    "diene_max_neighbor_charge", "diene_min_neighbor_charge", "diene_max_charge", "diene_min_charge", "bond_1", "bond2", "bond3", "distance_ad",
    "angle_a_cos", "angle_b_cos", "diene_max_angle_cos", "diene_min_angle_cos", "torsion",
    "diene_homo-1", "diene_homo", "diene_lumo", "diene_lumo+1", "diene_dipole", 
    "max_diene_L", "min_diene_L", "max_diene_B1", "min_diene_B1", "max_diene_B5", "min_diene_B5"]
ALL_PROPERTIES_ene = ["ene_charge_e", "ene_charge_f", "ene_max_neighbor_charge", "ene_min_neighbor_charge", 
    "diene_max_charge", "diene_min_charge", "bond4", 
    "ene_max_angle_cos", "ene_min_angle_cos", "ene_homo-1", "ene_homo", "ene_lumo", "ene_lumo+1", 
    "ene_dipole", "max_ene_L", "min_ene_L", "max_ene_B1", "min_ene_B1", "max_ene_B5", "min_ene_B5"]

In [ ]:
smiles_dict=pd.read_csv(SMILES_DIR + "/" + "all_smiles.csv")
smiles_dir=SMILES_DIR
main.smiles_property_generate(file_name='Descriptors.csv', smiles_dict=smiles_dict, smiles_dir=smiles_dir)

### Calculate SVO

In [ ]:
new_des = ['diene_area_0', 'diene_area_1', 'diene_area_2', 'diene_area_3', 'diene_area_4', 'diene_area_5', 'diene_area_6', 'diene_area_7', 
'ene_area_0', 'ene_area_1', 'ene_area_2', 'ene_area_3', 'ene_area_4', 'ene_area_5', 'ene_area_6', 'ene_area_7', ]

In [ ]:
df = pd.read_csv(SMILES_DIR + "/" + "Descriptors.csv")
all_properties = [[] for _ in range(len(new_des))]
mol_dir=SMILES_DIR
for idx, row in tqdm(df.iterrows()):
    smiles = row["Smiles"]
    smiles_id = row["Smiles_Id"]
    smiles_type = row['Type']
    title = row['Title']
    title = [int(each) for each in title.split()]
    mol = Chem.MolFromMolFile(mol_dir + '/mol/smilesid_%.5d.mol' % smiles_id, removeHs=False)
    if smiles_type == 'Diene':
        # try:
        for mol, atom_lists, tran_cis in cycle_process.change_position(mol, prop='diene', return_tran_cis=True):
            if atom_lists[0] == title[0] and atom_lists[-1] == title[1]:
                # result = [tran_cis]
                result = model_train.Calc_areas(mol, atom_lists)
                break
        assert len(result) == 8
        [all_properties[each].append(result[each]) for each in range(0,8)]
        [all_properties[each].append(np.nan) for each in range(8,16)]
        # except:
        #     print(smiles_id, title)
    else:
        for mol, atom_lists in cycle_process.change_position(mol, prop='dieno'):
            if atom_lists[0] == title[0] and atom_lists[-1] == title[1]:
                result = model_train.Calc_areas(mol, atom_lists)
                break
            if atom_lists[0] == title[1] and atom_lists[-1] == title[0]:
                result = model_train.Calc_areas(mol, atom_lists)
                result = [result[5], result[4], result[7], result[6], result[1], result[0], result[3], result[2]]
                break
        [all_properties[each].append(np.nan) for each in range(0,8)]
        [all_properties[each].append(result[each - 8]) for each in range(8,16)]
for each_new_des, all_property in zip(new_des, all_properties):
    df[each_new_des] = all_property
df.to_csv(SMILES_DIR + "/" + "Descriptors_.csv", index=False)

In [ ]:
import pickle
smiles_dir = SMILES_DIR
property_dict=pd.read_csv(SMILES_DIR + "/" + "Descriptors_.csv")
# smiles_dict = 
target_map = {}
for idx in range(len(property_dict["Smiles"])):
    smiles_id = property_dict["Smiles_Id"][idx]
    type_name = property_dict["Type"][idx]
    if type_name == "Diene":
        type_id = 1
    elif type_name == "Ene":
        type_id = 0
    title = property_dict["Title"][idx]
    key = "%.5d %d " % (smiles_id, type_id) + title
    if type_id == 1:
        result = [property_dict[property][idx] for property in main.ALL_PROPERTIES_diene + new_des[:8]]
    elif type_id == 0:
        result = [property_dict[property][idx] for property in main.ALL_PROPERTIES_ene + new_des[8:]]

    target_map[key] = result
with open(smiles_dir + "/" + "Descriptors_.pkl", "wb") as f:
    pickle.dump(target_map, f)